# Experiment: Training Baselines with Leakage-Safe Splits

Objective:
- Build two quick training baselines for next-day particle mass from wildfire raster samples.
- Baseline A: point model using per-day spatial averages for each channel.
- Baseline B: full-image model using downsampled image tensors.

Key requirement:
- Prevent leakage with both temporal and geospatial constraints at the fire level.


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zarr

from gribcheck.config import load_config

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
plt.style.use('seaborn-v0_8-whitegrid')

SEED = 7
np.random.seed(SEED)
SEED


## Plan

- Load index + Zarr artifacts and align to a consistent sample range.
- Build fire-level temporal splits, then enforce geospatial holdout across splits.
- Train/evaluate a point baseline (`channel mean/std -> next-day mean MASSDEN`).
- Train/evaluate a fast full-image model on downsampled patches.
- Compare metrics across train/val/test.


In [ ]:
# Load index metadata and Zarr dataset references
PROJECT_ROOT = Path.cwd()
cfg = load_config(PROJECT_ROOT / 'config/pipeline_config.yaml')

index_parquet_path = cfg.paths.wildfire_index_output
index_checkpoint_path = index_parquet_path.with_suffix('.checkpoint.jsonl')
zarr_path = cfg.paths.wildfire_zarr_output

if index_parquet_path.exists():
    raw_index_df = pd.read_parquet(index_parquet_path)
    index_source = 'parquet'
elif index_checkpoint_path.exists():
    raw_index_df = pd.read_json(index_checkpoint_path, lines=True)
    index_source = 'checkpoint'
else:
    raise FileNotFoundError(
        f'No wildfire index found at {index_parquet_path} or {index_checkpoint_path}'
    )

if not zarr_path.exists():
    raise FileNotFoundError(f'Wildfire zarr not found: {zarr_path}')

# Keep latest row per sample_key/sample_id to avoid resume duplicates.
index_df = raw_index_df.copy()
if 'sample_id' in index_df.columns:
    index_df = index_df.sort_values('sample_id').drop_duplicates(subset=['sample_id'], keep='last')
if 'sample_key' in index_df.columns:
    index_df = index_df.drop_duplicates(subset=['sample_key'], keep='last')
index_df = index_df.reset_index(drop=True)

required_cols = {
    'sample_id',
    'fire_id',
    'run_date',
    'bbox_min_lon',
    'bbox_min_lat',
    'bbox_max_lon',
    'bbox_max_lat',
}
missing_cols = required_cols.difference(index_df.columns)
if missing_cols:
    raise ValueError(f'Index missing columns: {sorted(missing_cols)}')

index_df['sample_id'] = index_df['sample_id'].astype(int)
index_df['run_date'] = pd.to_datetime(index_df['run_date'])
index_df['center_lat'] = (index_df['bbox_min_lat'] + index_df['bbox_max_lat']) / 2.0
index_df['center_lon'] = (index_df['bbox_min_lon'] + index_df['bbox_max_lon']) / 2.0

zg = zarr.open_group(str(zarr_path), mode='r')
input_channels = list(zg['inputs'].array_keys())
label_name = 't_plus_24h'
if label_name not in zg['labels'].array_keys():
    raise ValueError(f'Missing label array labels/{label_name}')

# Align to safe common length in case arrays are partially written.
available_n = min(
    [zg[f'inputs/{ch}'].shape[0] for ch in input_channels]
    + [zg[f'labels/{label_name}'].shape[0]]
)

index_df = index_df[index_df['sample_id'] < available_n].copy().reset_index(drop=True)

artifact_summary = pd.Series(
    {
        'index_source': index_source,
        'index_rows_after_dedupe': int(len(index_df)),
        'unique_fires': int(index_df['fire_id'].nunique()),
        'sample_id_min': int(index_df['sample_id'].min()),
        'sample_id_max': int(index_df['sample_id'].max()),
        'zarr_common_sample_count': int(available_n),
        'channel_count': int(len(input_channels)),
        'run_date_min': index_df['run_date'].min(),
        'run_date_max': index_df['run_date'].max(),
    },
    name='value',
)

artifact_summary.to_frame()


In [ ]:
# Build leakage-safe geospatial + temporal splits at fire level
@dataclass
class SplitConfig:
    train_quantile: float = 0.70
    val_quantile: float = 0.85
    geo_cell_deg: float = 0.25


def build_fire_splits(df: pd.DataFrame, cfg_split: SplitConfig) -> tuple[pd.DataFrame, dict[str, Any]]:
    fire_meta = (
        df.groupby('fire_id', as_index=False)
        .agg(
            fire_start=('run_date', 'min'),
            fire_end=('run_date', 'max'),
            center_lat=('center_lat', 'mean'),
            center_lon=('center_lon', 'mean'),
            n_samples=('sample_id', 'size'),
        )
        .sort_values(['fire_start', 'fire_id'])
        .reset_index(drop=True)
    )

    q_train = fire_meta['fire_start'].quantile(cfg_split.train_quantile)
    q_val = fire_meta['fire_start'].quantile(cfg_split.val_quantile)

    fire_meta['geo_lat_bin'] = (fire_meta['center_lat'] / cfg_split.geo_cell_deg).round().astype(int)
    fire_meta['geo_lon_bin'] = (fire_meta['center_lon'] / cfg_split.geo_cell_deg).round().astype(int)
    fire_meta['geo_cell'] = fire_meta['geo_lat_bin'].astype(str) + '_' + fire_meta['geo_lon_bin'].astype(str)

    train_fires = fire_meta[fire_meta['fire_start'] <= q_train].copy()
    val_fires = fire_meta[(fire_meta['fire_start'] > q_train) & (fire_meta['fire_start'] <= q_val)].copy()
    test_fires = fire_meta[fire_meta['fire_start'] > q_val].copy()

    train_fires['split_clean'] = 'train'
    val_fires['split_clean'] = 'val'
    test_fires['split_clean'] = 'test'

    # Geo holdout: later splits cannot reuse geo cells seen in earlier splits.
    train_geo = set(train_fires['geo_cell'])
    val_fires = val_fires[~val_fires['geo_cell'].isin(train_geo)].copy()
    val_geo = set(val_fires['geo_cell'])
    test_fires = test_fires[~test_fires['geo_cell'].isin(train_geo | val_geo)].copy()

    split_fire_df = pd.concat([train_fires, val_fires, test_fires], ignore_index=True)

    summary = {
        'q_train': q_train,
        'q_val': q_val,
        'fire_counts': split_fire_df['split_clean'].value_counts().to_dict(),
        'geo_cells_by_split': split_fire_df.groupby('split_clean')['geo_cell'].nunique().to_dict(),
        'dropped_fire_count_due_geo': int(len(fire_meta) - len(split_fire_df)),
    }
    return split_fire_df, summary


split_cfg = SplitConfig(train_quantile=0.70, val_quantile=0.85, geo_cell_deg=0.25)
split_fire_df, split_summary = build_fire_splits(index_df, split_cfg)

sample_df = index_df.merge(
    split_fire_df[['fire_id', 'split_clean', 'geo_cell', 'fire_start']],
    on='fire_id',
    how='inner',
)

sample_df = sample_df.sort_values('sample_id').reset_index(drop=True)

# Leakage checks.
train_fire_ids = set(sample_df.loc[sample_df['split_clean'] == 'train', 'fire_id'])
val_fire_ids = set(sample_df.loc[sample_df['split_clean'] == 'val', 'fire_id'])
test_fire_ids = set(sample_df.loc[sample_df['split_clean'] == 'test', 'fire_id'])
assert train_fire_ids.isdisjoint(val_fire_ids)
assert train_fire_ids.isdisjoint(test_fire_ids)
assert val_fire_ids.isdisjoint(test_fire_ids)

train_geo = set(sample_df.loc[sample_df['split_clean'] == 'train', 'geo_cell'])
val_geo = set(sample_df.loc[sample_df['split_clean'] == 'val', 'geo_cell'])
test_geo = set(sample_df.loc[sample_df['split_clean'] == 'test', 'geo_cell'])
assert train_geo.isdisjoint(val_geo)
assert train_geo.isdisjoint(test_geo)
assert val_geo.isdisjoint(test_geo)

split_counts = (
    sample_df.groupby('split_clean', as_index=False)
    .agg(
        samples=('sample_id', 'size'),
        fires=('fire_id', 'nunique'),
        geo_cells=('geo_cell', 'nunique'),
        run_date_min=('run_date', 'min'),
        run_date_max=('run_date', 'max'),
    )
    .sort_values('split_clean')
    .reset_index(drop=True)
)

print('Split summary:', split_summary)
split_counts


In [ ]:
# Visual check for temporal and geospatial split separation
fire_plot_df = split_fire_df.copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'train': '#1f77b4', 'val': '#ff7f0e', 'test': '#2ca02c'}
for split_name, grp in fire_plot_df.groupby('split_clean'):
    axes[0].scatter(grp['center_lon'], grp['center_lat'], s=28, alpha=0.75, label=split_name, color=colors.get(split_name, '#333333'))
axes[0].set_title('Fire Centroids by Split (Geo Holdout)')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].legend()

fire_timeline = (
    fire_plot_df.groupby(['fire_start', 'split_clean'], as_index=False)
    .size()
    .rename(columns={'size': 'fires'})
    .sort_values('fire_start')
)
for split_name, grp in fire_timeline.groupby('split_clean'):
    axes[1].plot(grp['fire_start'], grp['fires'], marker='o', linewidth=1.0, label=split_name, color=colors.get(split_name, '#333333'))
axes[1].set_title('Fire Start Timeline by Split')
axes[1].set_xlabel('Fire start date')
axes[1].set_ylabel('Number of fires')
axes[1].legend()

plt.tight_layout()
plt.show()


## Experiment A: Point Baseline (Daily Spatial Averages)

For each sample/day, collapse each input channel to a scalar via spatial mean and std,
then predict the next-day average particle mass (`labels/t_plus_24h` patch mean).


In [ ]:
# Build target and point-level features with date-aware unit harmonization + target transform
# HRRR smoke note: MASSDEN switched from ug/m^3 to kg/m^3 on 2021-12-21.
MASSDEN_UNIT_CUTOVER_UTC = pd.Timestamp('2021-12-21T00:00:00Z')
KG_TO_UG = 1e9


def massden_to_ugm3_date_aware(values: np.ndarray, run_dates_utc: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    run_dates_utc = pd.to_datetime(run_dates_utc, utc=True)
    scale = np.where(run_dates_utc < MASSDEN_UNIT_CUTOVER_UTC, 1.0, KG_TO_UG).astype(np.float32)
    return values * scale


def forward_target_transform(y_raw_ugm3: np.ndarray) -> np.ndarray:
    # Stabilize long-tailed smoke distribution.
    y_pos = np.clip(np.asarray(y_raw_ugm3, dtype=np.float32), a_min=0.0, a_max=None)
    return np.log1p(y_pos)


def inverse_target_transform(y_model_space: np.ndarray) -> np.ndarray:
    return np.expm1(np.asarray(y_model_space, dtype=np.float32))


sample_ids = sample_df['sample_id'].to_numpy(dtype=int)
split_labels = sample_df['split_clean'].to_numpy()
sample_run_dates_utc = pd.to_datetime(sample_df['run_date'], utc=True).to_numpy()

# Stored label values as in zarr.
y_all_stored = np.asarray(zg[f'labels/{label_name}'][:available_n], dtype=np.float32).mean(axis=(1, 2))
y_point_stored = y_all_stored[sample_ids].astype(np.float32)

y_point_raw_ugm3 = massden_to_ugm3_date_aware(y_point_stored, sample_run_dates_utc)
y_point = forward_target_transform(y_point_raw_ugm3)

point_feature_blocks = []
point_feature_names = []

for channel in input_channels:
    arr = np.asarray(zg[f'inputs/{channel}'][:available_n], dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    # Build features only for the working sample subset to avoid large in-memory transforms.
    arr_subset = arr[sample_ids]
    feat_mean = arr_subset.mean(axis=(1, 2))
    feat_std = arr_subset.std(axis=(1, 2))

    if channel.startswith('MASSDEN'):
        feat_mean = massden_to_ugm3_date_aware(feat_mean, sample_run_dates_utc)
        feat_std = massden_to_ugm3_date_aware(feat_std, sample_run_dates_utc)
        feat_mean = np.log1p(np.clip(feat_mean, a_min=0.0, a_max=None))
        feat_std = np.log1p(np.clip(feat_std, a_min=0.0, a_max=None))

    point_feature_blocks.append(feat_mean)
    point_feature_blocks.append(feat_std)
    point_feature_names.append(f'{channel}__mean')
    point_feature_names.append(f'{channel}__std')

X_point = np.column_stack(point_feature_blocks).astype(np.float32)

pd.Series(
    {
        'point_feature_count': int(X_point.shape[1]),
        'samples_used': int(X_point.shape[0]),
        'unit_cutover_utc': str(MASSDEN_UNIT_CUTOVER_UTC),
        'target_raw_mean_ugm3': float(np.mean(y_point_raw_ugm3)),
        'target_raw_std_ugm3': float(np.std(y_point_raw_ugm3)),
        'target_raw_p95_ugm3': float(np.quantile(y_point_raw_ugm3, 0.95)),
        'target_raw_p99_ugm3': float(np.quantile(y_point_raw_ugm3, 0.99)),
        'target_log_mean': float(np.mean(y_point)),
        'target_log_std': float(np.std(y_point)),
    },
    name='value',
).to_frame()


In [ ]:
# Helpers for regression fitting/evaluation
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    sst = float(np.sum((y_true - y_true.mean()) ** 2))
    sse = float(np.sum((y_true - y_pred) ** 2))
    if sst < 1e-12:
        r2 = np.nan
    else:
        r2 = float(1.0 - (sse / sst))
    return {'rmse': rmse, 'mae': mae, 'r2': r2}


def fit_ridge_primal(
    X: np.ndarray,
    y_model: np.ndarray,
    y_raw: np.ndarray,
    split_labels: np.ndarray,
    l2: float = 1.0,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    train_mask = split_labels == 'train'
    if train_mask.sum() == 0:
        raise ValueError('No train samples available.')

    mu = X[train_mask].mean(axis=0)
    sigma = X[train_mask].std(axis=0) + 1e-6
    Xn = (X - mu) / sigma

    y_mean = float(y_model[train_mask].mean())
    y_center = y_model - y_mean

    xtx = Xn[train_mask].T @ Xn[train_mask]
    xty = Xn[train_mask].T @ y_center[train_mask]
    w = np.linalg.solve(xtx + l2 * np.eye(xtx.shape[0], dtype=np.float64), xty)

    y_pred_model = (Xn @ w) + y_mean
    y_pred_raw = inverse_target_transform(y_pred_model)

    rows = []
    for split in ['train', 'val', 'test']:
        mask = split_labels == split
        if int(mask.sum()) == 0:
            continue
        m_raw = regression_metrics(y_raw[mask], y_pred_raw[mask])
        m_log = regression_metrics(y_model[mask], y_pred_model[mask])
        rows.append(
            {
                'split': split,
                'n': int(mask.sum()),
                'rmse': m_raw['rmse'],
                'mae': m_raw['mae'],
                'r2': m_raw['r2'],
                'rmse_log': m_log['rmse'],
                'mae_log': m_log['mae'],
                'r2_log': m_log['r2'],
            }
        )

    return pd.DataFrame(rows), y_pred_model.astype(np.float32), y_pred_raw.astype(np.float32)


point_metrics_df, point_pred_model, point_pred_raw = fit_ridge_primal(
    X=X_point,
    y_model=y_point,
    y_raw=y_point_raw_ugm3,
    split_labels=split_labels,
    l2=1.0,
)
point_metrics_df


In [ ]:
# Point baseline diagnostic plots (reported in raw ug/m^3)
point_eval_df = sample_df[['sample_id', 'fire_id', 'run_date', 'split_clean']].copy()
point_eval_df['y_true_ugm3'] = y_point_raw_ugm3
point_eval_df['y_pred_point_ugm3'] = point_pred_raw
point_eval_df['residual_point_ugm3'] = point_eval_df['y_pred_point_ugm3'] - point_eval_df['y_true_ugm3']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for split in ['train', 'val', 'test']:
    d = point_eval_df[point_eval_df['split_clean'] == split]
    if d.empty:
        continue
    axes[0].scatter(d['y_true_ugm3'], d['y_pred_point_ugm3'], s=10, alpha=0.45, label=split)
min_v = float(point_eval_df['y_true_ugm3'].min())
max_v = float(point_eval_df['y_true_ugm3'].max())
axes[0].plot([min_v, max_v], [min_v, max_v], linestyle='--', color='black', linewidth=1)
axes[0].set_title('Point Baseline: Pred vs True (ug/m^3)')
axes[0].set_xlabel('True next-day mean MASSDEN (ug/m^3)')
axes[0].set_ylabel('Predicted (ug/m^3)')
axes[0].legend()

axes[1].hist(point_eval_df['residual_point_ugm3'], bins=50, color='#ff7f0e')
axes[1].set_title('Point Baseline Residuals (ug/m^3)')
axes[1].set_xlabel('Prediction - True')

plt.tight_layout()
plt.show()


## Experiment B: Full-Image Fast Model

Use full image patches (downsampled to `32x32` for speed) and predict next-day average particle mass.

Model paths:
- If `torch` is available: train a tiny CNN on MPS/CPU.
- Otherwise: fallback to a fast ridge regressor on flattened image tensors.


In [ ]:
# Build downsampled full-image tensors
IMAGE_CHANNELS = ['VIIRS_FRP_1h', 'MASSDEN_8m', 'UGRD_10m', 'VGRD_10m']
missing_img_channels = [c for c in IMAGE_CHANNELS if c not in input_channels]
if missing_img_channels:
    raise ValueError(f'Missing image channels in zarr: {missing_img_channels}')

POOL_FACTOR = 8  # 256 -> 32


def downsample_mean(arr: np.ndarray, factor: int) -> np.ndarray:
    n, h, w = arr.shape
    if h % factor != 0 or w % factor != 0:
        raise ValueError(f'Cannot downsample shape {(h, w)} by factor {factor}')
    arr = arr.reshape(n, h // factor, factor, w // factor, factor)
    return arr.mean(axis=(2, 4))


image_blocks = []
for channel in IMAGE_CHANNELS:
    arr = np.asarray(zg[f'inputs/{channel}'][:available_n], dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr_ds = downsample_mean(arr, POOL_FACTOR)
    image_blocks.append(arr_ds)

X_img_all = np.stack(image_blocks, axis=1).astype(np.float32)  # [N, C, H, W]
X_img = X_img_all[sample_ids]

# Date-aware harmonization for MASSDEN channel only, then log1p.
massden_channel_idx = IMAGE_CHANNELS.index('MASSDEN_8m')
X_massden = X_img[:, massden_channel_idx, :, :]
scale = np.where(pd.to_datetime(sample_run_dates_utc, utc=True) < MASSDEN_UNIT_CUTOVER_UTC, 1.0, KG_TO_UG).astype(np.float32)
X_massden = X_massden * scale[:, None, None]
X_massden = np.log1p(np.clip(X_massden, a_min=0.0, a_max=None))
X_img[:, massden_channel_idx, :, :] = X_massden

pd.Series(
    {
        'image_tensor_shape': str(tuple(X_img.shape)),
        'channels_used': ', '.join(IMAGE_CHANNELS),
        'pool_factor': POOL_FACTOR,
        'massden_input_transform': 'date-aware units -> ug/m^3, then log1p',
    },
    name='value',
).to_frame()


In [ ]:
# Train full-image model (torch tiny CNN if available, else numpy dual-ridge)
def fit_ridge_dual(
    X: np.ndarray,
    y_model: np.ndarray,
    y_raw: np.ndarray,
    split_labels: np.ndarray,
    l2: float = 10.0,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    train_mask = split_labels == 'train'
    if train_mask.sum() == 0:
        raise ValueError('No train samples available for image model.')

    mu = X[train_mask].mean(axis=0)
    sigma = X[train_mask].std(axis=0) + 1e-6
    Xn = (X - mu) / sigma

    y_mean = float(y_model[train_mask].mean())
    y_center = y_model - y_mean

    x_train = Xn[train_mask]
    y_train = y_center[train_mask]

    kernel = x_train @ x_train.T
    alpha = np.linalg.solve(kernel + l2 * np.eye(kernel.shape[0], dtype=np.float64), y_train)
    y_pred_model = (Xn @ x_train.T @ alpha) + y_mean
    y_pred_raw = inverse_target_transform(y_pred_model)

    rows = []
    for split in ['train', 'val', 'test']:
        mask = split_labels == split
        if int(mask.sum()) == 0:
            continue
        m_raw = regression_metrics(y_raw[mask], y_pred_raw[mask])
        m_log = regression_metrics(y_model[mask], y_pred_model[mask])
        rows.append(
            {
                'split': split,
                'n': int(mask.sum()),
                'rmse': m_raw['rmse'],
                'mae': m_raw['mae'],
                'r2': m_raw['r2'],
                'rmse_log': m_log['rmse'],
                'mae_log': m_log['mae'],
                'r2_log': m_log['r2'],
            }
        )

    return pd.DataFrame(rows), y_pred_model.astype(np.float32), y_pred_raw.astype(np.float32)


image_model_name = 'numpy_ridge_flat_image'
image_pred_model: np.ndarray
image_pred_raw: np.ndarray

try:
    import torch
    import torch.nn as nn

    torch_available = True
except Exception:
    torch_available = False

if torch_available:
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

    class TinyCNNRegressor(nn.Module):
        def __init__(self, in_channels: int):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(in_channels, 16, kernel_size=5, stride=2, padding=2),
                nn.ReLU(),
                nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
                nn.ReLU(),
                nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1),
                nn.ReLU(),
                nn.AdaptiveAvgPool2d(1),
            )
            self.head = nn.Linear(32, 1)

        def forward(self, x):
            x = self.net(x)
            x = x.view(x.shape[0], -1)
            return self.head(x).squeeze(-1)

    train_mask = split_labels == 'train'
    val_mask = split_labels == 'val'
    test_mask = split_labels == 'test'

    x_train = X_img[train_mask]
    y_train_model = y_point[train_mask]

    channel_mu = x_train.mean(axis=(0, 2, 3), keepdims=True)
    channel_sigma = x_train.std(axis=(0, 2, 3), keepdims=True) + 1e-6
    Xn_img = (X_img - channel_mu) / channel_sigma

    model = TinyCNNRegressor(in_channels=X_img.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    loss_fn = nn.MSELoss()

    x_train_t = torch.tensor(Xn_img[train_mask], dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train_model, dtype=torch.float32, device=device)

    epochs = 8
    batch_size = 64

    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(x_train_t.shape[0], device=device)
        epoch_loss = 0.0
        for start in range(0, x_train_t.shape[0], batch_size):
            idx_batch = perm[start:start + batch_size]
            xb = x_train_t[idx_batch]
            yb = y_train_t[idx_batch]

            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

            epoch_loss += float(loss.item()) * float(xb.shape[0])

        epoch_loss /= float(x_train_t.shape[0])
        if epoch in (0, epochs - 1):
            print(f'Epoch {epoch + 1}/{epochs} - train MSE (log space): {epoch_loss:.6f}')

    model.eval()
    with torch.no_grad():
        x_all_t = torch.tensor(Xn_img, dtype=torch.float32, device=device)
        image_pred_model = model(x_all_t).cpu().numpy().astype(np.float32)

    image_pred_raw = inverse_target_transform(image_pred_model)
    image_model_name = f'tiny_cnn_torch_{device.type}'

    image_rows = []
    for split, mask in [('train', train_mask), ('val', val_mask), ('test', test_mask)]:
        if int(mask.sum()) == 0:
            continue
        m_raw = regression_metrics(y_point_raw_ugm3[mask], image_pred_raw[mask])
        m_log = regression_metrics(y_point[mask], image_pred_model[mask])
        image_rows.append(
            {
                'split': split,
                'n': int(mask.sum()),
                'rmse': m_raw['rmse'],
                'mae': m_raw['mae'],
                'r2': m_raw['r2'],
                'rmse_log': m_log['rmse'],
                'mae_log': m_log['mae'],
                'r2_log': m_log['r2'],
            }
        )
    image_metrics_df = pd.DataFrame(image_rows)

else:
    X_img_flat = X_img.reshape(X_img.shape[0], -1)
    image_metrics_df, image_pred_model, image_pred_raw = fit_ridge_dual(
        X=X_img_flat,
        y_model=y_point,
        y_raw=y_point_raw_ugm3,
        split_labels=split_labels,
        l2=10.0,
    )

print('Image model path:', image_model_name)
image_metrics_df


In [ ]:
# Compare both experiments
point_cmp = point_metrics_df.copy()
point_cmp['model'] = 'point_ridge_daily_avg'

image_cmp = image_metrics_df.copy()
image_cmp['model'] = image_model_name

comparison_df = pd.concat([point_cmp, image_cmp], ignore_index=True)
comparison_df = comparison_df[
    ['model', 'split', 'n', 'rmse', 'mae', 'r2', 'rmse_log', 'mae_log', 'r2_log']
].sort_values(['split', 'model']).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

rmse_plot = comparison_df.pivot(index='split', columns='model', values='rmse')
mae_plot = comparison_df.pivot(index='split', columns='model', values='mae')

rmse_plot.plot(kind='bar', ax=axes[0])
axes[0].set_title('RMSE by Split (ug/m^3)')
axes[0].set_ylabel('RMSE (ug/m^3)')
axes[0].set_xlabel('Split')

mae_plot.plot(kind='bar', ax=axes[1])
axes[1].set_title('MAE by Split (ug/m^3)')
axes[1].set_ylabel('MAE (ug/m^3)')
axes[1].set_xlabel('Split')

plt.tight_layout()
plt.show()

comparison_df


## Notes

- This notebook enforces leakage constraints at the fire level with both temporal windows and geo-cell holdout.
- The point model is intentionally minimal for quick sanity checking.
- The full-image section defaults to a fast NumPy model in this environment; if `torch` is installed, it will run a tiny CNN instead.
- PM2.5-specific modeling is intentionally left for the separate notebook.
